In [9]:
# ======= Variables =========

# Fill in the variables below and run cell first
# Note: Use forward slashes / instead of backslashes \ in file paths

DATASET_DIR = "../dataset"
OUTPUT_DIR = f"{DATASET_DIR}/original_sheets"

MODEL_TABLE_FILENAME = "model_table.csv"
DIAGNOSIS_DATASET_FILENAME = "diagnosis_dataset.csv"
OUTPUT_MODEL_TABLE_FILENAME = "model_table_feature_engineered.csv"

TRAINING_SET_CSV_FILE_PATH = f"{DATASET_DIR}/training_set.csv"
TEST_SET_CSV_FILE_PATH = f"{DATASET_DIR}/test_set.csv"
TEST_SET_SIZE = 0.1  # fraction of the dataset to be used as test set

MODEL_FOLDER_PATH = "../models"

RANDOM_SEED = 123

In [7]:
# ========= Libraries =========

import pandas as pd
import numpy as np
import sklearn as sk
import matplotlib.pyplot as plt


In [ ]:
# ========= Create Training and Test sets =========

# Load dataset into a pandas DataFrame
df = pd.read_csv(f"{DATASET_DIR}/{OUTPUT_MODEL_TABLE_FILENAME}")

# ============== Preprocessing steps ==============
df = df[df["HIGH_RISK"] == 1]  # Keep only the high-risk patients
df = df.drop(columns=["PATIENT_ID", "DISEASEX_DT"]) # Drop columns that will not be used as features for training the model

# TODO: Implement one-hot encoding for categorical features

# Separate a Test set and keep it aside
dfTest = df.sample(frac=TEST_SET_SIZE, axis='index', random_state=RANDOM_SEED) # random_state parameter is included for reproducibility
df = df.drop(dfTest.index) # the remaining data is used as the training set

# Save the Training and Test sets to CSV files
df.to_csv(TRAINING_SET_CSV_FILE_PATH, index=False)
dfTest.to_csv(TEST_SET_CSV_FILE_PATH, index=False)

In [11]:
# ========= Load Training and Test sets =========

# Load the Training set into a pandas dataframe
dfTraining = pd.read_csv(TRAINING_SET_CSV_FILE_PATH)

# Load the Test set into a pandas dataframe
dfTest = pd.read_csv(TEST_SET_CSV_FILE_PATH)


In [ ]:
# ========= Prepare Training and Test sets as Arrays =========

TARGET_COLUMN = "TARGET"

X_train = dfTraining.drop(columns=[TARGET_COLUMN]).to_numpy()
y_train = dfTraining[TARGET_COLUMN].to_numpy()

X_test = dfTest.drop(columns=[TARGET_COLUMN]).to_numpy()
y_test = dfTest[TARGET_COLUMN].to_numpy()

In [ ]:
# ========= Functions for metrics and plots =========

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def compute_metrics(y_true, y_pred):
    """
    Compute regression metrics between y_true and y_pred.
    Returns a dict with keys: MAE, RMSE, R2, Corr.
    """
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "Corr": corr}

def print_metrics(metrics, label=""):
    """
    Nicely print out a metrics dict from compute_metrics.
    """
    prefix = f"{label} " if label else ""
    print(f"{prefix}→ MAE: {metrics['MAE']:.2f}, "
          f"RMSE: {metrics['RMSE']:.2f}, "
          f"R²: {metrics['R2']:.3f}, "
          f"Corr: {metrics['Corr']:.3f}")

def plot_actual_vs_predicted(y_true, y_pred, set_name=""):
    """
    Scatter-plot actual vs. predicted with a y=x reference line.
    """
    plt.figure(figsize=(6,6))
    plt.scatter(y_true, y_pred, alpha=0.4, s=1, marker='o')
    lims = [min(y_true.min(),  y_pred.min()),
            max(y_true.max(),  y_pred.max())]
    plt.plot(lims, lims, 'r--', linewidth=1)
    plt.xlabel(f'Actual {TARGET_COLUMN}')
    plt.ylabel(f'Predicted {TARGET_COLUMN}')
    title = f'Actual vs Predicted ({set_name})' if set_name else 'Actual vs Predicted'
    plt.title(title)
    plt.tight_layout()
    plt.show()
    

In [ ]:
# ========= Train the Random Forest model =========

import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

N_TREES = 100           # Number of trees in the forest

# --- Train the model ---
model = RandomForestRegressor(
    n_estimators=N_TREES,    # Number of trees in the forest
    max_depth=None,     # Maximum depth of the tree (None means nodes are expanded until all leaves are pure)
    random_state=42,    # Random seed for reproducibility
    n_jobs=-1           # -1 value means to use all available cores
)
model.fit(X_train, y_train)

# --- Predict ---
y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

# --- Compute & print metrics ---
train_metrics = compute_metrics(y_train, y_train_pred)
test_metrics  = compute_metrics(y_test,  y_test_pred)

print_metrics(train_metrics, label="TRAIN")
print_metrics(test_metrics,  label=" TEST")

# --- Plot ---
plot_actual_vs_predicted(y_train, y_train_pred, set_name="Training Set")
plot_actual_vs_predicted(y_test,  y_test_pred,  set_name="Test Set")

In [ ]:
# ========= Save model =========
import joblib

def getCurrentTimestampString():
    """ Get current timestamp in the format of YYYYMMDD-HHMMSS """
    from datetime import datetime
    return datetime.now().strftime("%Y%m%d-%H%M%S")

joblib.dump(model, f'{MODEL_FOLDER_PATH}/{getCurrentTimestampString()}_random_forest_model.joblib', compress=3)
print("Model saved successfully!")

In [ ]:
# ========= Load model =========
import joblib

MODEL_FILENAME = "20250426-220333_random_forest_model.joblib"  # Update this to the actual filename of the model you want to load

# Update the path manually
model = joblib.load(f'{MODEL_FOLDER_PATH}/{MODEL_FILENAME}')
print("Model loaded successfully!")

In [ ]:
# --- Predict ---
y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

# --- Compute & print metrics ---
train_metrics = compute_metrics(y_train, y_train_pred)
test_metrics  = compute_metrics(y_test,  y_test_pred)

print_metrics(train_metrics, label="TRAIN")
print_metrics(test_metrics,  label=" TEST")

# --- Plot ---
plot_actual_vs_predicted(y_train, y_train_pred, set_name="Training Set")
plot_actual_vs_predicted(y_test,  y_test_pred,  set_name="Test Set")